<a href="https://colab.research.google.com/github/epicariello/zone30/blob/main/Zone30.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

import pandas as pd
import numpy as np
import re

# File path
file_path = '/content/Incidenti, morti e feriti - comuni.xlsx'

# Carica dati dalla riga 8 (salta 7 righe), NO header
df = pd.read_excel(file_path, skiprows=7, header=None)

# Crea nomi colonne: Comune + morti/feriti/incidenti per 2001-2024
anni = range(2001, 2025)
colonne = ['Comune']
for anno in anni:
    colonne.extend([f'{anno}_morti', f'{anno}_feriti', f'{anno}_incidenti'])

df.columns = colonne

# 🔧 PULIZIA ROBUSTA per risolvere l'errore '..'
def pulisci_valori(val):
    if pd.isna(val):
        return 0
    val_str = str(val).strip()
    # Rimuovi '..', ',', e non numeri
    if val_str == '..' or val_str == 'nan':
        return 0
    # Rimuovi virgole per numeri italiani
    val_str = val_str.replace(',', '')
    try:
        return int(float(val_str))
    except (ValueError, TypeError):
        return 0

# Applica pulizia a TUTTE le colonne numeriche
colonne_num = [col for col in df.columns if col != 'Comune']
for col in colonne_num:
    df[col] = df[col].apply(pulisci_valori)

# Pulisci 'Comune'
df['Comune'] = df['Comune'].astype(str).str.strip()
df = df[df['Comune'].str.len() > 0].reset_index(drop=True)

print(f"✅ DataFrame 'df' pronto! Shape: {df.shape}")
print("\nPrime righe:")
print(df[['Comune', '2001_morti', '2001_feriti', '2001_incidenti',
          '2024_morti', '2024_feriti', '2024_incidenti']].head())

print("\n📊 Pronto per analisi!")


✅ DataFrame 'df' pronto! Shape: (8578, 73)

Prime righe:
            Comune  2001_morti  2001_feriti  2001_incidenti  2024_morti  \
0            Agliè           0           10               5           0   
1          Airasca           0           48              25           0   
2     Ala di Stura           0            0               0           0   
3  Albiano d'Ivrea           1            6               5           0   
4  Alice Superiore           4            3               3           0   

   2024_feriti  2024_incidenti  
0            5               2  
1            9               5  
2            2               1  
3            9               4  
4            0               0  

📊 Pronto per analisi!


In [ ]:
import pandas as pd
import numpy as np

# Carica Comuni Emilia-Romagna
df_comuni_emilia_romagna = pd.read_excel('/content/Comuni_Emilia_Romagna.xlsx')

print("✅ df_comuni_emilia_romagna pronto!")
print(f"Shape: {df_comuni_emilia_romagna.shape}")
print("\nColonne:")
print(df_comuni_emilia_romagna.columns.tolist())
print("\nPrime righe:")
print(df_comuni_emilia_romagna.head())

print("\n📊 Riepilogo:")
print(df_comuni_emilia_romagna['Provincia'].value_counts())
print(f"\nTotale comuni: {len(df_comuni_emilia_romagna)}")


✅ df_comuni_emilia_romagna pronto!
Shape: (330, 4)

Colonne:
['Numero', 'Comune', 'Provincia', 'Popolazione (Censimento 2011)']

Prime righe:
   Numero     Comune      Provincia  Popolazione (Censimento 2011)
0       1   Agazzano       Piacenza                           2070
1       2   Albareto          Parma                           2165
2       3    Albinea  Reggio Emilia                           8755
3       4  Alfonsine        Ravenna                          12245
4       5     Alseno       Piacenza                           4823

📊 Riepilogo:
Provincia
Bologna          55
Modena           47
Piacenza         46
Parma            44
Reggio Emilia    42
Forlì-Cesena     30
Rimini           27
Ferrara          21
Ravenna          18
Name: count, dtype: int64

Totale comuni: 330


In [ ]:
# Comuni presenti in df_comuni_emilia_romagna ma ASSENTI in df (incidenti)
comuni_emilia_non_incidenti = set(df_comuni_emilia_romagna['Comune'].str.lower()) - set(df['Comune'].str.lower())

print("🔍 **COMUNI EMILIA-ROMAGNA ASSENTI negli incidenti:**")
print(f"Totale: {len(comuni_emilia_non_incidenti)}")
if len(comuni_emilia_non_incidenti) > 0:
    print(list(comuni_emilia_non_incidenti)[:20])  # Primi 20
    if len(comuni_emilia_non_incidenti) > 20:
        print("... e altri", len(comuni_emilia_non_incidenti)-20)
else:
    print("✅ Tutti i comuni Emilia-Romagna sono presenti!")

print("\n📊 Riepilogo:")
print(f"Comuni Emilia-Romagna totali: {len(df_comuni_emilia_romagna)}")
print(f"Comuni con incidenti: {len(set(df['Comune'].str.lower()) & set(df_comuni_emilia_romagna['Comune'].str.lower()))}")
print(f"Copertura: {100*(1-len(comuni_emilia_non_incidenti)/len(df_comuni_emilia_romagna)):.1f}%")


🔍 **COMUNI EMILIA-ROMAGNA ASSENTI negli incidenti:**
Totale: 0
✅ Tutti i comuni Emilia-Romagna sono presenti!

📊 Riepilogo:
Comuni Emilia-Romagna totali: 330
Comuni con incidenti: 330
Copertura: 100.0%


In [ ]:
import pandas as pd
import numpy as np

# Carica Comuni Sardegna
df_comuni_sardegna = pd.read_excel('Comuni_Sardegna.xlsx')

print("✅ df_comuni_sardegna pronto!")
print(f"Shape: {df_comuni_sardegna.shape}")
print("\nColonne:")
print(df_comuni_sardegna.columns.tolist())
print("\nPrime righe:")
print(df_comuni_sardegna.head())

print("\n📊 Riepilogo:")
print(df_comuni_sardegna['Provincia'].value_counts())
print(f"\nTotale comuni: {len(df_comuni_sardegna)}")


✅ df_comuni_sardegna pronto!
Shape: (377, 5)

Colonne:
['Comune', 'Provincia', 'Popolazione (2021)', 'Altitudine (m)', 'Superficie (km²)']

Prime righe:
          Comune                  Provincia  Popolazione (2021)  \
0      Abbasanta                   Oristano                2579   
1         Aggius  Gallura Nord-Est Sardegna                1409   
2       Aglientu  Gallura Nord-Est Sardegna                1154   
3   Aidomaggiore                   Oristano                 398   
4  Alà dei Sardi  Gallura Nord-Est Sardegna                1764   

   Altitudine (m)  Superficie (km²)  
0             315             39.85  
1             514             86.31  
2             420            148.19  
3             250             41.21  
4             663            197.99  

📊 Riepilogo:
Provincia
Oristano                     87
Cagliari                     70
Sassari                      66
Nuoro                        53
Medio Campidano              28
Gallura Nord-Est Sardegna    26


In [ ]:
# Verifica comuni SARDAGNA nel df incidenti
comuni_sardegna = set(df_comuni_sardegna['Comune'].str.lower())
comuni_incidenti = set(df['Comune'].str.lower())

comuni_sard_non_incidenti = comuni_sardegna - comuni_incidenti

print("🔍 **COMUNI SARDAGNA ASSENTI negli incidenti:**")
print(f"Totale: {len(comuni_sard_non_incidenti)}")
if len(comuni_sard_non_incidenti) > 0:
    print("\nPrimi 20:")
    for i, comune in enumerate(sorted(comuni_sard_non_incidenti)[:20]):
        print(f"  {i+1}. {comune.title()}")
    if len(comuni_sard_non_incidenti) > 20:
        print(f"\n... e altri {len(comuni_sard_non_incidenti)-20}")
else:
    print("✅ Tutti i comuni Sardegna SONO presenti!")

print(f"\n📊 Copertura:")
print(f"Comuni Sardegna totali: {len(df_comuni_sardegna):,}")
print(f"Comuni con incidenti: {len(comuni_sardegna & comuni_incidenti):,}")
print(f"Copertura: {100*(len(comuni_sardegna & comuni_incidenti)/len(df_comuni_sardegna)):.1f}%")


🔍 **COMUNI SARDAGNA ASSENTI negli incidenti:**
Totale: 0
✅ Tutti i comuni Sardegna SONO presenti!

📊 Copertura:
Comuni Sardegna totali: 377
Comuni con incidenti: 377
Copertura: 100.0%


In [ ]:
import pandas as pd

# Carica il CSV della popolazione per comune per anno
df_pop = pd.read_csv(
    '/content/drive/MyDrive/TesiMagistrale/FontiUsate/popolazione_comuni_per_anno.csv',
    sep=';',
    decimal=','
)

# Rinomina per chiarezza
df_pop = df_pop.rename(columns={'codice': 'Codice', 'comune': 'Comune'})

print("✅ df_pop pronto!")
print(f"Shape: {df_pop.shape}")
print("\nColonne:")
print(df_pop.columns.tolist())
print("\nPrime righe:")
print(df_pop.head())

# -------------------------------------------------------
# Verifica: tutti i comuni di df sono presenti in df_pop?
# -------------------------------------------------------
comuni_df = set(df['Comune'].str.strip().str.lower())
comuni_pop = set(df_pop['Comune'].str.strip().str.lower())

assenti_in_pop = comuni_df - comuni_pop

print("\n🔍 Comuni presenti in df (incidenti) ma ASSENTI in df_pop (popolazione):")
print(f"Totale: {len(assenti_in_pop)}")
if assenti_in_pop:
    for c in sorted(assenti_in_pop):
        print(f"  - {c.title()}")
else:
    print("✅ Tutti i comuni di df sono presenti in df_pop!")

print(f"\n📊 Riepilogo copertura:")
print(f"Comuni in df (incidenti):   {len(comuni_df):,}")
print(f"Comuni in df_pop:            {len(comuni_pop):,}")
print(f"Comuni coperti:              {len(comuni_df & comuni_pop):,}")
print(f"Copertura: {100 * len(comuni_df & comuni_pop) / len(comuni_df):.1f}%")
